# 🧪 Curiosity On Demand - End-to-End Agent Testing

This notebook allows you to test, inspect, and benchmark the full autonomous video generation pipeline:
1. **`ResearcherAgent`:** Search-grounded text investigation using Gemini with Google Search.
2. **`EvaluatorAgent`:** Quality evaluation, filtering, and structured `ResearchReport` generation in an iterative multi-turn loop.
3. **`ScriptWriterAgent`:** Dynamic scene budgeting (~15–25s per scene based on requested video duration) with scene-by-scene educational narration and rich visual choreography.
4. **`NarratorAgent` (`NarrationPlannerAgent` & `NarrationGeneratorAgent`):** Voice casting from 30 Gemini TTS voices, directorial delivery tuning, inline emotional audio tags (`[whispers]`, `[excitedly]`), and high-fidelity speech synthesis (`gemini-3.1-flash-tts-preview`).
5. **`AnimationPlannerAgent`:** Visual style design, color palette selection, typography, and motion guidelines.
6. **`SceneAnimatorAgent`:** Executable HTML5 Canvas animation generation with 2D virtual camera transforms (dolly/pan/zoom) and multi-act visual phase progression.
7. **`AnimationRenderer`:** Headless Chromium rendering of sample frame snapshots (PNG), parallel real-time MP4 video assembly, and audio/video multiplexing with FFmpeg.
8. **`CostTracker`:** Real-time token usage, search grounding query counting, speech audio duration, and USD cost tracking across every pipeline component.

## 1. Setup & Environment Configuration

In [ ]:
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, Audio, Image, Video, display

# Ensure project root is in sys.path when running from notebooks/
project_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables (.env)
load_dotenv(project_root / ".env")

from agents.animation_planner import AnimationPlannerAgent
from agents.evaluator import EvaluatorAgent
from agents.narrator import NarratorAgent
from agents.researcher import ResearcherAgent
from agents.scene_animator import SceneAnimatorAgent
from agents.script_writer import ScriptWriterAgent
from config.settings import get_settings
from core.cost_tracker import (
    CostTracker,
    PipelineComponent,
    set_global_cost_tracker,
    track_component,
)
from core.database import get_db_manager
from core.models import (
    AnimationStoryboard,
    EvaluationResult,
    NarrationStoryboard,
    ResearchReport,
    Script,
)
from services.renderer import AnimationRenderer

settings = get_settings()

# Initialize Database and CostTracker
db = get_db_manager()
cost_tracker = CostTracker(db_manager=db)
set_global_cost_tracker(cost_tracker)

print("✅ Environment initialized successfully.")
print(f"• Gemini Model:     {settings.gemini_model}")
print(f"• Research Model:   {settings.research_model}")
print(f"• Evaluator Model:  {settings.evaluator_model}")
print(f"• Script Model:     {settings.script_model}")
print(f"• TTS Planner:      {settings.narration_planner_model}")
print(f"• TTS Model:        {settings.tts_model}")
print(f"• Default Voice:    {settings.default_tts_voice}")
print(f"• Animator Model:   {settings.animator_model}")
print(f"• Video Resolution: {settings.video_width}x{settings.video_height}")
print(
    f"• API Key Set:      {'Yes' if settings.gemini_api_key else 'No (using default credentials)'}"
)
print(
    "• Cost Tracker:     Active (tracking tokens, search queries, audio duration & costs)"
)

## 2. Define Topic & Target Duration

In [ ]:
import re
import uuid
from datetime import UTC, datetime

# TOPIC = "The Cthulu mythos"
TOPIC = "The Dinossaurs eras"
TARGET_DURATION = "~2 minutes"

# Unique project run subfolder based on topic slug and timestamp
topic_slug = re.sub(r"[^\w]+", "_", TOPIC.strip().lower()).strip("_") or "project"
timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
project_folder = f"{topic_slug}_{timestamp}"

# Unique project ID for non-invasive cost and component attribution
PROJECT_ID = f"notebook_{timestamp}_{uuid.uuid4().hex[:6]}"

# Project-specific output directories under storage/
audio_dir = settings.audio_dir / project_folder
render_dir = settings.renders_dir / project_folder
anim_dir = settings.animations_dir / project_folder

audio_dir.mkdir(parents=True, exist_ok=True)
render_dir.mkdir(parents=True, exist_ok=True)
anim_dir.mkdir(parents=True, exist_ok=True)

print(f"🎯 Selected Topic:   '{TOPIC}' (Target Duration: {TARGET_DURATION})")
print(f"🆔 Project ID:        {PROJECT_ID}")
print(f"📁 Project Subfolder: {project_folder}")
print(f"• Audio Output:       {audio_dir}")
print(f"• Render Output:      {render_dir}")
print(f"• Animation Output:   {anim_dir}")

## 3. Run Iterative Research & Evaluation Loop

In [ ]:
researcher = ResearcherAgent(settings=settings)
evaluator = EvaluatorAgent(settings=settings)

accumulated_notes = ""
feedback = None
final_report: ResearchReport | None = None
max_iterations = settings.max_research_iterations

for iteration in range(1, max_iterations + 1):
    print(f"\n🔄 --- Iteration Round {iteration}/{max_iterations} ---")
    print("🔍 Researcher gathering search-grounded facts...")
    with track_component(PROJECT_ID, PipelineComponent.RESEARCH):
        round_notes = await researcher.conduct_research(
            topic=TOPIC,
            feedback=feedback,
            previous_research=accumulated_notes if accumulated_notes else None,
            target_duration=TARGET_DURATION,
        )

    accumulated_notes = (
        f"{accumulated_notes}\n\n=== ROUND {iteration} ===\n{round_notes}"
        if accumulated_notes
        else round_notes
    )

    print("⚖️ Evaluator assessing completeness and structuring schema...")
    with track_component(PROJECT_ID, PipelineComponent.EVALUATION):
        eval_result: EvaluationResult = await evaluator.evaluate_research(
            topic=TOPIC,
            accumulated_research=accumulated_notes,
            target_duration=TARGET_DURATION,
            iteration=iteration,
            max_iterations=max_iterations,
        )

    final_report = eval_result.report

    if eval_result.is_sufficient or iteration >= max_iterations:
        print(f"✅ Research finalized! Sufficient: {eval_result.is_sufficient}")
        break

    feedback = eval_result.feedback_for_researcher
    print(f"⚠️ Content incomplete. Evaluator feedback for next round:\n{feedback}")

## 4. Inspect Researched Report

In [ ]:
if final_report:
    print(f"📋 TOPIC: {final_report.topic}\n")
    print(f"💡 SUMMARY:\n{final_report.summary}\n")
    print("=" * 60 + "\n")
    print(f"📚 RESEARCHED SUBTOPICS ({len(final_report.sections)} sections):\n")
    for i, section in enumerate(final_report.sections, 1):
        print(f"[{i}] 📌 {section.topic}")
        print(f"{section.content}")
        print("-" * 60)

## 5. Generate Scene-by-Scene Narration Script (Dynamic Scene Budgeting)
Use `ScriptWriterAgent` to convert the structured research into an educational narration script.
The agent dynamically budgets scene counts and durations based on the target video length (~15–25 seconds per scene, ~35–55 words each) to ensure dynamic pacing and frequent visual cuts.

In [ ]:
assert final_report is not None, "Final report must be generated before script writing"
from agents.script_writer import calculate_scene_budget, parse_duration_to_seconds

script_writer = ScriptWriterAgent(settings=settings)

target_seconds = parse_duration_to_seconds(TARGET_DURATION)
min_scenes, max_scenes, avg_scene_dur = calculate_scene_budget(target_seconds)
print(f"📊 Dynamic Scene Budget for '{TARGET_DURATION}' (~{target_seconds}s total):")
print(f"   • Expected Scenes: {min_scenes} to {max_scenes}")
print(f"   • Target Duration: ~{avg_scene_dur}s per scene (~35-55 words)\n")

print("✍️ Generating scene-by-scene narration script...")
with track_component(PROJECT_ID, PipelineComponent.SCRIPT_WRITING):
    generated_script: Script = await script_writer.generate_script(
        topic=TOPIC,
        research_report=final_report,
        target_duration=TARGET_DURATION,
    )

total_seconds = sum(s.duration_seconds for s in generated_script.scenes)
print(
    f"✅ Script Generated: '{generated_script.topic}' ({len(generated_script.scenes)} scenes, Total Budgeted: {total_seconds}s / {total_seconds // 60}m {total_seconds % 60}s)"
)
print(f"📝 Overview: {generated_script.summary}\n")
print("=" * 60 + "\n")

for scene in generated_script.scenes:
    word_count = len(scene.narration.split())
    print(
        f"🎬 Scene {scene.scene_number}: {scene.title} (Duration: {scene.duration_seconds}s, Words: {word_count})"
    )
    print(f"📌 Summary:            {scene.summary}")
    print(f"🎨 Visual Description: {scene.description}")
    print(f"🎙️ Narration:          {scene.narration}")
    print("-" * 60)

## 6. Formulate Narration Plan & Synthesize Speech Audio (Gemini TTS)
Use `NarratorAgent` (coordinating `NarrationPlannerAgent` and `NarrationGeneratorAgent`) to:
1. Cast a voice from the 30 supported Gemini TTS voices and define character personas and studio acoustic vibe.
2. Enrich script narration with inline emotional and pacing audio tags (e.g. `[whispers]`, `[excitedly]`, `[amazed]`, `[gasp]`, `[thoughtful]`, `[very fast]`).
3. Synthesize natural-sounding speech audio via `gemini-3.1-flash-tts-preview` in 24kHz 16-bit PCM RIFF WAV format.

In [ ]:
narrator = NarratorAgent(settings=settings)
print(
    "🎙️ Formulating directorial narration plan and synthesizing Scene 1 voice sample..."
)

with track_component(PROJECT_ID, PipelineComponent.NARRATION_PLANNING):
    narration_plan, scene_1_audio = await narrator.determine_plan_and_sample(
        topic=TOPIC,
        script=generated_script,
        output_dir=audio_dir,
    )

print(f"✅ Voice Cast:       {narration_plan.voice_name}")
print(
    f"🎭 Character Role:   {narration_plan.audio_profile_name} — {narration_plan.audio_profile_role}"
)
print(f"🎬 Scene Vibe:       {narration_plan.scene_environment}")
print(f"📋 Director Notes:   {narration_plan.directors_notes}")
print("=" * 60)

scene_1_plan = narration_plan.scenes[0]
print("🎬 Scene 1 Original Narration:")
print(f"   {scene_1_plan.original_narration}\n")
print("🎧 Scene 1 Enhanced Narration (with Audio Tags):")
print(f"   {scene_1_plan.enhanced_narration}\n")
print(
    f"⏱️ Sample Audio Duration: {scene_1_audio.duration_seconds:.2f}s ({scene_1_audio.file_size_bytes} bytes)"
)
print(f"📁 Audio File Path:       {scene_1_audio.audio_path}\n")

print("▶️ Play Scene 1 Voice Preview:")
display(Audio(filename=scene_1_audio.audio_path))

### 6.1 Synthesize Full Narration Storyboard (All Scenes in Parallel)
Generate speech audio clips for all script scenes concurrently and combine them into a continuous master audio track (`narration_full.wav`).

In [ ]:
print(
    f"🎙️ Synthesizing narration audio for all {len(narration_plan.scenes)} scenes in parallel..."
)

with track_component(PROJECT_ID, PipelineComponent.VOICE_SYNTHESIS):
    narration_storyboard: NarrationStoryboard = await narrator.generate_full_narration(
        topic=TOPIC,
        plan=narration_plan,
        output_dir=audio_dir,
        sample_scene_1=scene_1_audio,
        concurrency=3,
    )

print("✅ Full Narration Storyboard Synthesized!")
print(f"• Total Scenes:   {len(narration_storyboard.scenes_audio)}")
print(f"• Total Duration: {narration_storyboard.total_duration_seconds:.2f}s")
print(f"• Master Audio:   {narration_storyboard.full_audio_path}")
print("=" * 60)

for sa in narration_storyboard.scenes_audio:
    print(
        f"  🔊 Scene {sa.scene_number}: {sa.duration_seconds:.2f}s -> {sa.audio_path}"
    )

if narration_storyboard.full_audio_path:
    print("\n▶️ Play Master Narration Audio:")
    display(Audio(filename=narration_storyboard.full_audio_path))

## 7. Plan Visual Style & Generate Scene 1 HTML5 Canvas Code
Use `AnimationPlannerAgent` to formulate the visual design aesthetic and `SceneAnimatorAgent` to produce unconstrained executable HTML5 Canvas code for Scene 1.
The generated animation incorporates:
- **2D Virtual Camera & Viewport Choreography:** Dynamic zoom, pan, and dolly camera passes.
- **Multi-Act Visual Progression:** Phase 1 establishing entrance -> Phase 2 mechanism deep dive -> Phase 3 climax and metric callout.

In [ ]:
planner = AnimationPlannerAgent(settings=settings)
animator = SceneAnimatorAgent(settings=settings)

print("🎨 Planning visual animation style and design guidelines...")
with track_component(PROJECT_ID, PipelineComponent.STYLE_PLANNING):
    style = await planner.plan_style(
        topic=TOPIC,
        script=generated_script,
    )

print(f"✅ Visual Style: {style.style_name}")
print(f"🖌️ Palette:     {', '.join(style.color_palette)}")
print(f"🔤 Typography:  {style.font_family}")
print(f"📋 Description: {style.visual_description}")
print(f"💡 Guidelines:  {style.animation_guidelines}")
print("=" * 60)

s1_scene = generated_script.scenes[0]
s1_native_duration = (
    s1_scene.duration_seconds or settings.default_scene_duration_seconds
)

print(
    f"🎬 Generating unconstrained HTML5 Canvas code for Scene 1 (Native Duration: {s1_native_duration}s)..."
)
with track_component(PROJECT_ID, PipelineComponent.SCENE_ANIMATION):
    scene_1_anim = await animator.animate_scene(
        topic=TOPIC,
        scene=s1_scene,
        style=style,
        duration_seconds=s1_native_duration,
    )

scene_1_html_path = anim_dir / "scene_1.html"
scene_1_html_path.write_text(scene_1_anim.html_content, encoding="utf-8")
print(f"💾 Scene 1 HTML Code saved to: {scene_1_html_path}")
print(
    f"🎬 Scene 1 HTML Code Preview ({len(scene_1_anim.html_content)} chars, Duration: {scene_1_anim.duration_seconds}s):"
)

### 7.1 Interactive Animation Preview (Live Canvas Playback)
Inspect the live 60fps canvas animation with interactive motion, particles, and physics directly in the notebook before rendering snapshots or video clips.

In [ ]:
# Live in-notebook interactive animation playback
html_escaped = scene_1_anim.html_content.replace('"', "&quot;")
container_html = (
    "<div style='border: 2px solid #334155; border-radius: 8px; "
    "overflow: hidden; width: 100%; max-width: 800px; aspect-ratio: 16/9; "
    "background: #000; margin: 10px 0;'>"
    f"<iframe srcdoc='{html_escaped}' style='width: 100%; height: 100%; border: none;'></iframe>"
    "</div>"
)
display(HTML(container_html))

## 8. Render Scene 1 Sample Frame Snapshot (PNG)
Use `AnimationRenderer` with headless Chromium to render the Scene 1 HTML code and capture a high-resolution snapshot.

In [ ]:
renderer = AnimationRenderer(settings=settings)
sample_png_path = render_dir / "scene_1_sample.png"

print(f"📸 Rendering sample snapshot to: {sample_png_path}")
rendered_path = await renderer.render_sample_frame(
    html_content=scene_1_anim.html_content,
    output_image_path=sample_png_path,
)

print(f"✅ Snapshot rendered ({rendered_path.stat().st_size} bytes):")
display(Image(filename=str(rendered_path)))

## 9. Parallel Video Rendering, Audio Muxing & Master Video Assembly
Generate animation code for all scenes, render video clips **concurrently in parallel** using isolated browser contexts, assemble the final master MP4 video, and multiplex synchronized narration audio into the final output.

In [ ]:
narr_sb = locals().get("narration_storyboard")

print(
    f"🎬 Generating animation code for all {len(generated_script.scenes)} scenes in parallel..."
)
with track_component(PROJECT_ID, PipelineComponent.SCENE_ANIMATION):
    storyboard: AnimationStoryboard = await animator.animate_storyboard(
        topic=TOPIC,
        script=generated_script,
        style=style,
        sample_scene_1=scene_1_anim,
    )

# Persist HTML5 canvas animation code for each scene
for sc in storyboard.scenes:
    (anim_dir / f"scene_{sc.scene_number}.html").write_text(
        sc.html_content, encoding="utf-8"
    )
print(f"💾 Saved {len(storyboard.scenes)} scene HTML animation files to: {anim_dir}")

total_anim_duration = sum(s.duration_seconds for s in storyboard.scenes)
print()
print(
    f"🎥 Rendering {len(storyboard.scenes)} scene clips in parallel (Native Duration: {total_anim_duration:.2f}s)..."
)

start_time = time.perf_counter()
scene_video_paths = await renderer.render_scenes_parallel(
    scenes=storyboard.scenes,
    output_dir=render_dir,
    concurrency=3,
)
elapsed = time.perf_counter() - start_time
print(
    f"⚡ Parallel rendering completed in {elapsed:.1f}s (saved ~{max(0, total_anim_duration - elapsed):.1f}s vs sequential)!"
)

final_master_path = render_dir / "final_master_video.mp4"
print()
print(
    f"🎞️ Assembling {len(scene_video_paths)} scene clips with synchronized per-scene audio speed scaling..."
)
master_audio = (
    Path(narr_sb.full_audio_path)
    if narr_sb and getattr(narr_sb, "full_audio_path", None)
    else None
)
scene_audios = (
    [s.audio_path for s in narr_sb.scenes_audio if getattr(s, "audio_path", None)]
    if narr_sb and getattr(narr_sb, "scenes_audio", None)
    else None
)
await renderer.assemble_video(
    scene_video_paths=scene_video_paths,
    output_video_path=final_master_path,
    audio_path=master_audio,
    scene_audio_paths=scene_audios,
    narration_storyboard=narr_sb,
)

print(
    f"✅ Video Produced (with synchronized audio): {final_master_path} ({final_master_path.stat().st_size} bytes)"
)
display(Video(filename=str(final_master_path), embed=True))

## 10. Cost & Resource Consumption Summary
Inspect aggregated API token consumption, Google Search grounding query counts, speech audio synthesis duration, and estimated USD costs for this session across all pipeline stages.

In [ ]:
summary = await cost_tracker.get_project_summary(PROJECT_ID)

print(cost_tracker.format_telegram_report(summary))
print("\n" + "━" * 60)
print(f"🏷️ Caption Badge: {cost_tracker.format_short_badge(summary)}")

print("\n📋 Detailed Component Breakdown Table:")
header = f"{'Component':<26} | {'Model':<28} | {'Tokens (In / Out)':<22} | {'Searches':<9} | {'Audio':<8} | {'Cost (USD)':<10}"
print(header)
print("-" * len(header))
for c in summary.components:
    tok_str = f"{c.prompt_tokens:,} / {c.output_tokens:,}"
    audio_str = (
        f"{c.audio_duration_seconds:.1f}s" if c.audio_duration_seconds > 0 else "-"
    )
    searches_str = str(c.grounding_queries) if c.grounding_queries > 0 else "-"
    print(
        f"{c.component:<26} | {c.model_name:<28} | {tok_str:<22} | {searches_str:<9} | {audio_str:<8} | ${c.estimated_cost_usd:.5f}"
    )
print("-" * len(header))
print(
    f"{'TOTAL':<26} | {'':<28} | {f'{summary.total_prompt_tokens:,} / {summary.total_output_tokens:,}':<22} | {summary.total_grounding_queries:<9} | {f'{summary.total_audio_duration_seconds:.1f}s':<8} | ${summary.total_cost_usd:.4f}"
)